# §36 — Retention yasası LM ölçeğinde, cache confound'u kaldırılmış

**Otorite ön-kayıt `RESULTS.md` §36'da.** Bu markdown kolaylık kopyasıdır.

**Soru:** BEKLEYEN #21 raf koşulu #3 — cubic'in avantajı LM ölçeğinde tekrarlanıyor mu?

**§15h neden cevap değil:** §15h aynı ikizleri **KV-cache açıkken** karşılaştırdı.
Erratum (§30): bu hibritte uzun menzilli sinyali cache taşıyor, O(1) state değil.
State'e kör bir ölçüm, yalnız state üzerinde etkili olan retention yasasına da kördür.

**Tasarım — model-içi farkla state izolasyonu.** Cache her chunk sınırında sıfırlanır.
Dört koşul: cubic/exp × (state taşınıyor / state de sıfırlanıyor).
**State katkısı** Δ = CE(state sıfır) − CE(state taşınıyor). Model-içi fark, iki ikizin
farklı taban kalitesine sahip olması confound'unu kaldırır.

**Endpoint:** her chunk'ın ilk 32 tokenındaki ortalama CE (chunk ≥ 1), chunk'a göre
eşleşmiş. **Test önceden belirlendi: Wilcoxon** (§29b'de t-testi aykırılardan patladı).

**GÜÇ KAPISI:** iki kolda da Δ ≈ 0 ise state hiçbir şey taşımıyor demektir ve
karşılaştırma sıfır güçlüdür → **"sonuçsuz"** yazılır, "null" değil.

**Eşik:** |Δ_cubic − Δ_exp| ≥ 0.02 nat/token + Wilcoxon p < 0.05.

**Uyarı:** run5/run6 §15f/§15h dönemi YOĞUN graft'ı (~325k param, PPL ≈ 1.64×),
bugünkü 6-katman referansı değil. Bulgu o konfigürasyona ait olur.
## Checkpoint'ler nasil gelecek

`.gitignore` `*.pt` ve `checkpoints/` iceriyor -> **repo klonunda checkpoint YOK.**
Repoya commit ETME: `.pt` yayinlamak agirlik yayinlamaktir ve
`LISANS_KARAR_REHBERI.md` Kapi 2'yi tetikler (agirlik lisansi karari henuz
verilmedi, ve bir kez yayinlanan agirlik geri alinamaz).

Bunun yerine Kaggle'da **Add Data > Upload** ile ozel bir dataset olustur:

- `checkpoints/graft_run5/hfp_graft_final.pt` (cubic, ~1.35 MB)
- `checkpoints/graft_run6_exp/hfp_graft_exp_final.pt` (exp, ~1.35 MB)

Hucre 2 `/kaggle/input` altini arar, **parmak iziyle** dogrular ve Run 1'i
(out_gain ~0.75) reddeder — `KAYNAK.md`'de kayitli tuzak.


In [ ]:
# --- 1. KURULUM ---
import os, sys, glob, json, subprocess, math, time
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEV == 'cuda', 'GPU YOK. Accelerator > T4 sec.'
OUT = os.path.join(BASE,'v36'); os.makedirs(OUT, exist_ok=True)
print('repo:', REPO, '| cihaz:', torch.cuda.get_device_name(0), '| cikti:', OUT)

In [ ]:
# --- 2. IKIZ CHECKPOINTLERI BUL + PARMAK IZIYLE DOGRULA + KATMAN KUMESI ---
# NOT: checkpoint'ler .gitignore'da (*.pt, checkpoints/) -> repo klonunda YOKLAR.
# Kaggle: Add Data > Your Datasets ile iki .pt dosyasini yukle (bkz. markdown).
#
# DOSYA ADINA GUVENILMEZ. KAYNAK.md'deki ders: Run 1'in 'hfp_graft_final.pt'si
# bir ara 'run5' etiketiyle dolasti. Kimlik PARMAK IZIYLE dogrulanir:
#   Run 1  -> out_gain ort ~0.75   (YANLIS dosya)
#   Run 5  -> out_gain ort ~0.237  (cubic ikizi)
#   Run 6  -> out_gain ort ~0.239  (exp ikizi)
import re
ROOTS = ['/kaggle/input', REPO, BASE, '/content/drive/MyDrive', '/content', os.path.expanduser('~')]
cands = []
for r in ROOTS:
    if r and os.path.isdir(r): cands += glob.glob(f'{r}/**/*.pt', recursive=True)
cands = sorted(set(cands))
assert cands, ('HIC .pt DOSYASI YOK.\n'
               '  Kaggle: Add Data > Upload > iki dosyayi yukle:\n'
               '    checkpoints/graft_run5/hfp_graft_final.pt      (cubic)\n'
               '    checkpoints/graft_run6_exp/hfp_graft_exp_final.pt (exp)\n'
               '  Bunlar .gitignore\'da oldugu icin repo klonunda YOK.')

def probe(path):
    sd = torch.load(path, map_location='cpu')
    if isinstance(sd, dict) and 'm' in sd and isinstance(sd['m'], dict): sd = sd['m']
    og = [v.flatten() for k, v in sd.items() if k.endswith('out_gain')]
    if not og: return None
    og = torch.cat(og)
    ls = sorted({int(m.group(1)) for k in sd
                 for m in [re.search(r'layers\.(\d+)\.self_attn', k)] if m})
    return {'sd': sd, 'n': len(sd), 'og': float(og.mean()), 'ogsd': float(og.std()), 'layers': ls}

print(f"{'out_gain':>9} {'std':>7} {'tensor':>7} {'kat':>4}  dosya")
print('-'*84)
info = {}
for c in cands:
    p = probe(c)
    if p is None: continue
    info[c] = p
    flag = '  <-- Run1? (out_gain ~0.75, YANLIS)' if p['og'] > 0.5 else ''
    print(f"{p['og']:>9.4f} {p['ogsd']:>7.4f} {p['n']:>7} {len(p['layers']):>4}  {c}{flag}")
assert info, 'Bulunan .pt dosyalarinin hicbiri graft checkpointi degil (out_gain yok).'

# Secim: exp = dosya adinda 'exp'; cubic = 'exp' YOK. Ikisinde de parmak izi kapisi.
def pick(arm):
    want_exp = (arm == 'exp')
    hits = [c for c, p in info.items()
            if (('exp' in os.path.basename(c).lower()) == want_exp)
            and 'final' in os.path.basename(c).lower()
            and 0.15 <= p['og'] <= 0.40]          # PARMAK IZI KAPISI: Run1'i (~0.75) eler
    assert hits, (f'{arm} ikizi bulunamadi. Ya dosya yok, ya parmak izi disinda '
                  f'(0.15-0.40 bandi). Yukaridaki tabloya bak: Run1 (~0.75) KABUL EDILMEZ.')
    assert len(hits) == 1, f'{arm} icin BIRDEN FAZLA aday: {hits}. Fazlasini kaldir.'
    return hits[0]

CKPT = {a: pick(a) for a in ('cubic', 'exp')}
SD_CUBIC, SD_EXP = info[CKPT['cubic']]['sd'], info[CKPT['exp']]['sd']
L_CUBIC, L_EXP   = info[CKPT['cubic']]['layers'], info[CKPT['exp']]['layers']
print()
for a in ('cubic', 'exp'):
    i = info[CKPT[a]]
    print(f'{a:>6}: out_gain {i["og"]:.4f} | tensor {i["n"]} | katman {len(i["layers"])} | {CKPT[a]}')
assert L_CUBIC == L_EXP, ('IKIZ DEGILLER: katman kumeleri farkli -> tek-degisken '
                          f'kosulu ihlal, deney KOSULAMAZ.\n  cubic {L_CUBIC}\n  exp   {L_EXP}')
GRAFT_LAYERS = L_CUBIC
print(f'\nortak katman kumesi ({len(GRAFT_LAYERS)}): {GRAFT_LAYERS}')
print('KAYNAK.md referansi: Run5 out_gain ~0.237 (cubic), Run6 ~0.239 (exp)')


In [ ]:
# --- 3. BASE MODEL + GRAFT KURUCU (yukleme DOGRULAMASI zorunlu, §30 hucre 7) ---
import glob, os
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
from hfp.models.grafting import (GraftConfig, graft_llama, set_graft_mode,
                                 enable_streaming, reset_streaming, HFPGraftAttention)
ROOTS = ['/kaggle/input', REPO, BASE, os.path.expanduser('~'), '.']
def find_one(pat):
    hits = []
    for r in ROOTS:
        if r and os.path.isdir(r): hits += glob.glob(f'{r}/**/{pat}', recursive=True)
    return max(set(hits), key=os.path.getmtime) if hits else None

# Qwen: config.json ILE AGIRLIGIN AYNI KLASORDE oldugu yeri ara.
# Not: repodaki hf_upload/hf_release/config.json agirlik icermiyor; sadece
# 'config.json' aramak onu bulup SRC'yi bos birakiyordu.
_cands = []
for _r in ROOTS:
    if not (_r and os.path.isdir(_r)): continue
    for _cfg in glob.glob(f'{_r}/**/config.json', recursive=True):
        _d = os.path.dirname(_cfg)
        if glob.glob(f'{_d}/*.safetensors') or glob.glob(f'{_d}/*.bin'):
            _cands.append(_d)
assert _cands, ('QWEN YOK. Kaggle sag panel > Add Input > Models > Qwen2.5-1.5B.\n'
                '  (Datasets sekmesi degil, Models sekmesi.)')
SRC = _cands[0]
tok = AutoTokenizer.from_pretrained(SRC)
print('base model:', SRC)


def build(arm):
    """arm in {'cubic','exp'} -> yuklenmis, DOGRULANMIS grafted model."""
    mode = 'cubic_flux_chunked' if arm == 'cubic' else 'exp'
    m = AutoModelForCausalLM.from_pretrained(SRC, torch_dtype=torch.float32).to(DEV).eval()
    graft_llama(m, GraftConfig(decay_mode=mode, write_rule='hybrid',
                               key_feature_map='dpfp', rec_block=16), layers=GRAFT_LAYERS)
    for mm in m.modules():
        if isinstance(mm, HFPGraftAttention): mm.out_gain.data.fill_(0.1)
    sd = SD_CUBIC if arm == 'cubic' else SD_EXP
    m.load_state_dict(sd, strict=False)
    set_graft_mode(m, 'student'); m.config.use_cache = True
    # --- YUKLEME DOGRULAMASI (strict=False sessizce hicbir sey yuklemeyebilir) ---
    own = dict(m.state_dict())
    matched = [k for k in sd if k in own and own[k].shape == sd[k].shape]
    missing = [k for k in sd if k not in own]
    same = sum(1 for k in matched if torch.equal(own[k].to(DEV), sd[k].to(DEV)))
    og = torch.cat([mm.out_gain.detach().flatten() for mm in m.modules()
                    if isinstance(mm, HFPGraftAttention)])
    ok = (len(matched) == len(sd)) and (same == len(matched)) and (og.std() > 1e-4)
    print(f'[{arm}] tensor {len(sd)} | eslesen {len(matched)} | bit-bit ayni {same} | '
          f'eksik isim {len(missing)} | out_gain ort {og.mean():.4f} std {og.std():.4f}')
    assert ok, (f'[{arm}] CHECKPOINT YUKLENMEDI. out_gain std ~0 ve ort ~0.1 ise '
                f'agirliklar EGITIMSIZ -> hicbir sayi raporlanamaz. eksik: {missing[:3]}')
    print(f'[{arm}] YUKLEME DOGRULANDI')
    return m
print('\nkurucu hazir (build("cubic") / build("exp"))')

In [ ]:
# --- 4. DEGERLENDIRME METNI (held-out) ---
# S1 distilasyonu WikiText-103 TRAIN uzerinde yapildi -> VALIDATION held-out'tur.
CHUNK, HEAD, N_CHUNK = 256, 32, 120   # chunk uzunlugu | endpoint penceresi | chunk sayisi
try:
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='validation')
    text = '\n\n'.join(t for t in ds['text'] if len(t.strip()) > 200)
    kaynak = 'wikitext-103-raw-v1/validation'
except Exception as e:
    p = find_one('wiki.valid.tokens') or find_one('wiki.valid.raw')
    assert p, f'Degerlendirme metni yok ({type(e).__name__}). WikiText-103 valid ekle.'
    text = open(p, encoding='utf-8').read(); kaynak = p
ids = tok(text, return_tensors='pt').input_ids[0]
need = CHUNK * (N_CHUNK + 1)
assert ids.numel() >= need, f'metin kisa: {ids.numel()} < {need}'
ids = ids[:need]
print(f'kaynak: {kaynak}\ntoken: {ids.numel():,} | chunk {CHUNK} x {N_CHUNK+1} | endpoint = ilk {HEAD} token')

In [ ]:
# --- 5. KOSU: 2 kol x (state tasiniyor / state sifirlaniyor) ---
import torch.nn.functional as F

@torch.no_grad()
def run(m, carry_state):
    """Cache HER chunk sinirinda sifirlanir. carry_state=False ise HFP state'i de.
    Donen: chunk basina, ilk HEAD tokenin ortalama CE'si (chunk index >= 1)."""
    enable_streaming(m, True); reset_streaming(m)
    per_chunk = []
    for c in range(N_CHUNK + 1):
        x = ids[c*CHUNK:(c+1)*CHUNK].unsqueeze(0).to(DEV)
        if not carry_state: reset_streaming(m)        # KONTROL kolu
        past = DynamicCache()                          # cache HER chunk'ta sifir
        lg = m(x, past_key_values=past, use_cache=True).logits
        # HF kaydirmasi: logits[:, :-1] <-> x[:, 1:]. Ilk HEAD hedefi = x[1..HEAD]
        ce = F.cross_entropy(lg[0, :HEAD, :], x[0, 1:HEAD+1], reduction='mean').item()
        if c >= 1: per_chunk.append(ce)                # chunk 0'in oncesi yok
    return per_chunk

RES = {}
for arm in ('cubic', 'exp'):
    t0 = time.time(); m = build(arm)
    RES[arm] = {'carried': run(m, True), 'reset': run(m, False)}
    del m; torch.cuda.empty_cache()
    d = [r - c for r, c in zip(RES[arm]['reset'], RES[arm]['carried'])]
    print(f'[{arm}] CE tasiniyor {sum(RES[arm]["carried"])/len(d):.4f} | '
          f'sifir {sum(RES[arm]["reset"])/len(d):.4f} | '
          f'state katkisi D = {sum(d)/len(d):+.4f} nat/token  ({time.time()-t0:.0f}s)\n')
json.dump({'chunk':CHUNK,'head':HEAD,'n_chunk':N_CHUNK,'kaynak':kaynak,
           'layers':GRAFT_LAYERS,'res':RES}, open(f'{OUT}/v36_raw.json','w'), indent=2)
print('ham veri:', f'{OUT}/v36_raw.json')

In [ ]:
# --- 6. GUC KAPISI + ON-KAYITLI HUKUM (§36) ---
import statistics as st
def wilcoxon(d):
    dd = [x for x in d if x != 0]; n = len(dd)
    if n < 6: return float('nan'), float('nan')
    order = sorted(range(n), key=lambda i: abs(dd[i])); rank = [0.0]*n
    i = 0
    while i < n:
        j = i
        while j+1 < n and abs(dd[order[j+1]]) == abs(dd[order[i]]): j += 1
        avg = (i+j)/2 + 1
        for k in range(i, j+1): rank[order[k]] = avg
        i = j + 1
    W = sum(rank[i] for i in range(n) if dd[i] > 0)
    z = (W - n*(n+1)/4) / math.sqrt(n*(n+1)*(2*n+1)/24)
    return z, math.erfc(abs(z)/math.sqrt(2))

D = {a: [r-c for r, c in zip(RES[a]['reset'], RES[a]['carried'])] for a in ('cubic','exp')}
print('=== GUC KAPISI: state hic bir sey tasiyor mu? ===')
print('  (D = CE(state sifir) - CE(state tasiniyor); POZITIF = state yardim ediyor)')
gate = False
for a in ('cubic','exp'):
    z, p = wilcoxon(D[a]); m = st.mean(D[a])
    print(f'  {a:>6}: D = {m:+.4f} nat/token  (n={len(D[a])} chunk, Wilcoxon p={p:.4g})')
    if m > 0 and p < 0.05: gate = True
print(f'  -> KAPI {"GECILDI" if gate else "GECILMEDI"}')

print('\n=== ON-KAYITLI HUKUM (§36) ===')
if not gate:
    print('  SONUCSUZ (null DEGIL). Hicbir kolda O(1) state olculebilir bir katki')
    print('  yapmiyor -> cubic vs exp SIFIR GUCLU bir karsilastirma. Retention yasasi')
    print('  hakkinda hicbir iddia yazilmaz. Proje kurali geregi "etki yok" YAZILMAZ.')
    print('  On-kayitli durma kurali: cubic LM-olcek hatti KAPANIR.')
else:
    diff = [dc - de for dc, de in zip(D['cubic'], D['exp'])]
    md = st.mean(diff); z, p = wilcoxon(diff)
    print(f'  D_cubic - D_exp = {md:+.4f} nat/token  (n={len(diff)}, Wilcoxon z={z:+.2f} p={p:.4g})')
    print(f'  esik: |fark| >= 0.02 nat/token VE p < 0.05')
    if md >= 0.02 and p < 0.05:
        print('\n  => CUBIC AVANTAJI LM OLCEGINDE: cache confoundu kalkinca retention')
        print('     yasasi olculebilir sekilde onemli. §15h\'nin nullu protokolune atfedilir.')
        print('     Kosul #3 KARSILANDI. Tek izinli devam: guncel 6-katman receteyle tekrar.')
    elif md <= -0.02 and p < 0.05:
        print('\n  => TERS: exp burada da daha iyi. Cubic\'in LM-olcek savunmasi KAPANIR.')
    else:
        print('\n  => BERABERE: iki yasa bu konfigurasyonda LM olceginde denk.')
        print('     §15h ile birlikte IKINCI, protokol-duzeltilmis null -> kosul #3 KARSILANMADI.')
        print('     On-kayitli durma kurali: cubic LM-olcek hatti KAPANIR.')